In [ ]:
#Preparação do ambiente para rodar o Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/opt/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init('/opt/spark-3.3.0-bin-hadoop3')

from pyspark.sql import SparkSession

In [ ]:
# Sessão Spark
spark = SparkSession.builder \
    .appName("EcommerceDataLake") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/home/tavares/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

Criação da tabela principal

In [ ]:
spark.sql("""
CREATE TABLE hadoop_catalog.default.vendas_ecommerce (
    venda_id INT,
    produto_nome STRING,
    categoria STRING,
    quantidade INT,
    preco_unitario DOUBLE,
    data_venda DATE,
    cliente_id STRING,
    vendedor_id INT
)
USING iceberg
PARTITIONED BY (year(data_venda), categoria)
""").show()

In [ ]:
#Validação da tabela
spark.sql("""
    DESCRIBE FORMATTED hadoop_catalog.default.vendas_ecommerce
""").show(truncate=False)

Inserção de dados

In [ ]:
#Inserção de Dados Históricos 2023

spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (1, 'Notebook Dell', 'Eletrônicos', 2, 2500.00, DATE('2023-01-15'), 'CLI001', 101),
    (2, 'Mouse Logitech', 'Eletrônicos', 5, 80.00, DATE('2023-01-16'), 'CLI002', 102),
    (3, 'Mesa Escritório', 'Móveis', 1, 800.00, DATE('2023-02-10'), 'CLI003', 101),
    (4, 'Cadeira Gamer', 'Móveis', 2, 600.00, DATE('2023-02-15'), 'CLI001', 103),
    (5, 'Smartphone Samsung', 'Eletrônicos', 1, 1200.00, DATE('2023-03-20'), 'CLI004', 102)
""")

In [ ]:
#Inserção de dados 2024

spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (6, 'Tablet iPad', 'Eletrônicos', 1, 3000.00, DATE('2024-01-10'), 'CLI002', 101),
    (7, 'Sofá 3 Lugares', 'Móveis', 1, 1500.00, DATE('2024-01-20'), 'CLI005', 103),
    (8, 'Monitor 4K', 'Eletrônicos', 2, 800.00, DATE('2024-02-05'), 'CLI003', 102)
""")

Analise de Snapshots

In [ ]:
#Listando os snapshots criados ou seja ações realizadas na tabela
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce.snapshots").show()

In [ ]:
#Contando a quantidade de snapshots/ação na tabela
spark.sql("SELECT operation, COUNT(*) as num_operacoes FROM hadoop_catalog.default.vendas_ecommerce.snapshots GROUP BY operation;").show()

In [ ]:
#Visualizando historico da tabela
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce.history").show(truncate=False)

In [ ]:
#Acessando a primeira inserção de dados (snapshot)
snapshots = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at")
second_snapshot = snapshots.collect()[0][0]  # Segundo snapshot (índice 1)
print(f"ID do segundo snapshot: {second_snapshot}")

# Consultar dados como estavam no segundo snapshot
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF {second_snapshot}
    ORDER BY venda_id
""").show()

Time travel

In [ ]:
#Acessando a primeira inserção de dados (snapshot)
snapshots = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at")
second_snapshot = snapshots.collect()[0][0]  # Primeiro snapshot (índice o)
snapshot_atual = snapshots.collect()[3][0]  #snapshot atual (índice o)
print(f"ID do primeiro snapshot: {second_snapshot}")
print(f"ID do ultimo snapshot: {snapshot_atual}")

# Apenas os dados de 2023 usando o primeiro snapshot
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF {second_snapshot}
    ORDER BY venda_id
""").show()

# Apenas os dados de 2023 usando o primeiro snapshot
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF {snapshot_atual}
    ORDER BY venda_id
""").show()

Evolução de schema (add columns)

In [ ]:
# Adicionando novas colunas
spark.sql("""
    ALTER TABLE hadoop_catalog.default.vendas_ecommerce ADD COLUMNS desconto DOUBLE, canal_venda STRING
""")

In [ ]:
#Validando a dição das novas colunas
spark.sql("""
    DESCRIBE FORMATTED hadoop_catalog.default.vendas_ecommerce
""").show(truncate=False)

In [ ]:
#Inserção com Novo Schema
spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, DATE('2024-03-15'), 'CLI006', 101, 10.0, 'online'),
    (10, 'Mesa Centro', 'Móveis', 1, 400.00, DATE('2024-03-20'), 'CLI007', 102, 5.0, 'loja_fisica')
""").show(truncate=False)

In [ ]:
#Validando a inserção
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
""").show()

In [ ]:
#Mostre que dados antigos têm valores NULL nas novas colunas
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce where data_venda = '2023-01-15'
""").show()

Operações ACID e Merge

In [ ]:
#Atualizando o valor dos elestronicos
spark.sql(f"""
    UPDATE hadoop_catalog.default.vendas_ecommerce 
    SET preco_unitario = preco_unitario * 100 
    WHERE categoria = 'Eletrônicos'
""").show()

In [ ]:
#Validando a mudança de valores nos eletronicos
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
""").show()

Rollback

In [ ]:
Otimização e Análise